# Decision Trees, Random Forests and SVMs

Portfolio source extract. See the repository README and PROVENANCE.md for methods, data requirements, original evidence and publication changes. Run from the notebooks/ directory.

In [ ]:
import sys
# Publication: install dependencies from ../requirements.txt first.


In [ ]:
import sys
# Publication: install dependencies from ../requirements.txt first.


In [ ]:
# ==============================
# TO BE IMPLEMENTED
# ==============================

# import are relvant libraries
import sys
import pandas as pd
import numpy as np
import sklearn
import matplotlib
import seaborn
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader


In [ ]:
# Load training data
data = pd.read_csv("titanic.csv")

print(data.head())
print(data.info())
print(data.isnull().sum())



In [ ]:
data['Age'].isnull().sum()

In [ ]:
data['Age'].mean()

In [ ]:
data['Age'] = data['Age'].fillna(data['Age'].mean())

In [ ]:
data['Embarked'].isnull().sum()

In [ ]:
data['Embarked'].mode()

In [ ]:
data['Embarked'] = data['Embarked'].fillna(data['Embarked'].mode()[0])

In [ ]:
data['Embarked'].isnull().sum()

In [ ]:
df_dropped_irrelevant = data.drop(['Cabin', 'Ticket', 'Name', 'PassengerId'], axis=1)

print("Original DataFrame:")
print(data)
print("\nDataFrame after dropping 'Cabin', 'Ticket', 'Name' and 'PassengerId':")
print(df_dropped_irrelevant)
data = df_dropped_irrelevant

In [ ]:
data.head()

In [ ]:
df_dropped_irrelevant.head()

In [ ]:
cols = {'Sex': ['male', 'female', 'male', 'female'],
        'Embarked': ['S', 'C', 'Q', 'S']}
df = pd.DataFrame(cols)
df

In [ ]:

sex_mapping = {'male': 0, 'female': 1}
df['Sex'] = df['Sex'].map(sex_mapping)


embarked_mapping = {'C': 0, 'Q': 1, 'S': 2}
df['Embarked'] = df['Embarked'].map(embarked_mapping)

print(df)

In [ ]:
# ==============================
# TO BE IMPLEMENTED
# ==============================

# Handle missing values

# Drop unused columns

# Encode categorical features manually

sex_mapping = {'male': 0, 'female': 1}
data['Sex'] = data['Sex'].map(sex_mapping)


embarked_mapping = {'C': 0, 'Q': 1, 'S': 2}
data['Embarked'] = data['Embarked'].map(embarked_mapping)

In [ ]:
data['Embarked']

In [ ]:
data['Sex']

In [ ]:
X = data.drop('Survived', axis=1)
y = data['Survived'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
X_train, X_test, y_train, y_test

In [ ]:
# ==========================
# TO BE IMPLEMENTED
# ==========================

from pandas.api.types import is_numeric_dtype

def entropy(y):
    """
    Compute the entropy of a label array y.
    Formula: H(y) = -Σ p_i * log2(p_i)
    """

    _, counts = np.unique(y, return_counts=True)
    p_i = counts / counts.sum()

    h_y = - np.sum(p_i * np.log2(p_i + 1e-9))

    return h_y


def information_gain(y, y_left, y_right):
    """
    Compute the information gain of a split.
    IG = H(parent) - (n_left/n)*H(left) - (n_right/n)*H(right)
    """

    parent_entropy = entropy(y)

    n = len(y)
    n_left = len(y_left)
    n_right = len(y_right)

    entropy_left = entropy(y_left)
    entropy_right = entropy(y_right)

    IG = parent_entropy - (n_left/n) * entropy_left - (n_right / n) * entropy_right 
    
    return IG


def best_split(X, y, feature_subset=None):
    """
    Find the best feature and threshold that maximize information gain.

    Parameters:
        X (DataFrame): Feature dataset
        y (Series or array): Target labels
        feature_subset (list): Optional subset of features for random forest

    Returns:
        best_feature (str): Feature name with best split
        best_threshold (float/str): Threshold or category value
        best_gain (float): Information gain of best split
    """

    y = pd.Series(y, index=X.index)

    features = feature_subset if feature_subset is not None else list(X.columns)

    best_feature = None
    best_threshold = None
    best_gain = 0.0

    for feat in features:
        col = X[feat]
    
        if is_numeric_dtype(col):
            vals = np.unique(col.dropna().values)
            if len(vals) < 2:
                continue
            thr_list = (vals[:-1] + vals[1:]) / 2.0
            for t in thr_list:
                left_mask = col <= t
                right_mask = ~left_mask
                if left_mask.sum() == 0 or right_mask.sum() == 0:
                    continue
                gain = information_gain(y, y[left_mask], y[right_mask])
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feat
                    best_threshold = float(t)
        else:
            for val in pd.unique(col.dropna()):
                left_mask = col == val
                right_mask = ~left_mask
                if left_mask.sum() == 0 or right_mask.sum() == 0:
                    continue
                gain = information_gain(y, y[left_mask], y[right_mask])
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feat
                    best_threshold = val

    return best_feature, best_threshold, float(best_gain)


In [ ]:
entropy([0, 0, 1, 1])

In [ ]:
# Find best split on the training data
best_feature, best_threshold, best_gain = best_split(X_train, y_train)

print("Best feature:", best_feature)
print("Best threshold:", best_threshold)
print("Information gain:", best_gain)

# Create masks for the chosen split
if best_feature is not None:
    col = X_train[best_feature]
    if pd.api.types.is_numeric_dtype(col):
        left_mask  = col <= best_threshold
        right_mask = col >  best_threshold
    else:
        left_mask  = col == best_threshold
        right_mask = col != best_threshold

    # Split X and y accordingly (indices stay aligned)
    X_left,  y_left  = X_train[left_mask],  y_train[left_mask]
    X_right, y_right = X_train[right_mask], y_train[right_mask]

    print("Left size:", len(X_left), "Right size:", len(X_right))
else:
    print("No valid split found (node should be a leaf).")


In [ ]:

# ==============================
# TO BE IMPLEMENTED
# ==============================


import numpy as np
from collections import Counter

class Node:
    def __init__(
        self, feature=None, threshold=None, left=None, right=None, *, value=None
    ):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

    def is_leaf_node(self):
        return self.value is not None


class DecisionTree:
    def __init__(self, max_depth=5, min_samples_split=2, feature_subsample=None):
        """
        Initialize the Decision Tree parameters.
        """
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.feature_subsample = feature_subsample
        self.tree = None
        self._columns = None  # keep training column order/names for prediction

    def fit(self, X, y, depth=0):
        """
        Recursively build the decision tree using information gain.
        """
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)
        y = pd.Series(y, index=X.index)
        self._columns = list(X.columns)
        self.tree = self._grow_tree(X, y, depth=0)
        return self

    def predict_one(self, x, node=None):
        """
        Predict class label for a single sample.
        """
        node = self.tree if node is None else node
        # ensure we can access values by column name
        if not isinstance(x, (pd.Series, dict)):
            x = pd.Series(x, index=self._columns)

        while not node.is_leaf_node():
            val = x[node.feature]
            thr = node.threshold
            if self._is_numeric_threshold(thr):
                node = node.left if val <= thr else node.right
            else:
                node = node.left if val == thr else node.right
        return node.value

    def predict(self, X):
        """
        Predict class labels for all rows in dataset X.
        """
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X, columns=self._columns)
        return np.array([self.predict_one(row) for _, row in X.iterrows()])


    def _grow_tree(self, X, y, depth):
        if len(y) == 0:
            return Node(value=0)
        if len(np.unique(y)) == 1:
            return Node(value=y.iloc[0])
        if depth >= self.max_depth or len(y) < self.min_samples_split:
            return Node(value=self._majority(y))

        features = self._pick_features(X.columns)
        feat, thr, gain = best_split(X, y, feature_subset=features)
        if feat is None or gain <= 0:
            return Node(value=self._majority(y))

        col = X[feat]
        left_mask = (col <= thr) if self._is_numeric_threshold(thr) else (col == thr)
        right_mask = ~left_mask
        if left_mask.sum() == 0 or right_mask.sum() == 0:
            return Node(value=self._majority(y))

        left = self._grow_tree(X[left_mask], y[left_mask], depth + 1)
        right = self._grow_tree(X[right_mask], y[right_mask], depth + 1)
        return Node(feature=feat, threshold=thr, left=left, right=right)

    def _majority(self, y):
        return Counter(y).most_common(1)[0][0]

    def _pick_features(self, cols):
        cols = list(cols)
        if self.feature_subsample and self.feature_subsample < len(cols):
            return list(np.random.choice(cols, self.feature_subsample, replace=False))
        return cols

    def _is_numeric_threshold(self, thr):
        return isinstance(thr, (int, float, np.number))

In [ ]:
tree = DecisionTree(max_depth=4)
tree.fit(X_train, y_train)
y_pred_tree = tree.predict(X_test)

acc_tree = np.mean(y_pred_tree == y_test)
print(f"Custom Decision Tree Accuracy: {acc_tree:.4f}")

compare_tree = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred_tree})
compare_tree.head(10)


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from itertools import count
import numpy as np

def _node_text(n):
    if n.is_leaf_node():
        return f"Leaf: {n.value}"
    thr = n.threshold
    thr = f"{thr:.4f}" if isinstance(thr, (int, float, np.number)) else str(thr)
    op  = "<= " if isinstance(n.threshold, (int, float, np.number)) else "== "
    return f"{n.feature}\n{op}{thr}"

def _to_networkx(decision_tree):
    G = nx.DiGraph()
    c = count()

    def walk(n):
        nid = next(c)
        G.add_node(nid, label=_node_text(n))
        if not n.is_leaf_node():
            l_id = walk(n.left)
            r_id = walk(n.right)
            G.add_edge(nid, l_id, label="True")   # left: condition satisfied
            G.add_edge(nid, r_id, label="False")  # right: condition not satisfied
        return nid

    root_id = walk(decision_tree.tree)
    return G, root_id

def draw_tree(decision_tree, figsize=(10, 8)):
    import networkx as nx, matplotlib.pyplot as plt
    G, root = _to_networkx(decision_tree)   # from earlier code

    # --- the key line ---
    from networkx.drawing.nx_pydot import graphviz_layout
    pos = graphviz_layout(G, prog="dot")   # hierarchical layout (requires graphviz)
    # ---------------------

    plt.figure(figsize=figsize)
    nx.draw(G, pos, with_labels=False, node_size=1600, node_color="#DBEAFE", edgecolors="black")
    nx.draw_networkx_labels(G, pos, labels=nx.get_node_attributes(G, "label"), font_size=9)
    nx.draw_networkx_edge_labels(G, pos, edge_labels=nx.get_edge_attributes(G, "label"), font_size=8)
    plt.axis("off")
    plt.show()


draw_tree(tree)

In [ ]:
# ==============================
# TO BE IMPLEMENTED
# ==============================

import numpy as np
import pandas as pd
from collections import Counter

import numpy as np
from collections import Counter

class RandomForest:
    def __init__(self, n_estimators=10, max_depth=5, min_samples_split=2, feature_subsample_ratio=0.7):
        """
        Initialize the Random Forest parameters.
        """
        
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.feature_subsample_ratio = feature_subsample_ratio
        self.trees = []
        self._n_features_ = None
        self._feat_sub_count_ = None

    def _bootstrap_sample(self, X, y):
        """
        Row-wise bootstrap sample (works for DataFrame/Series/ndarray).
        """
        
        n = X.shape[0]
        idxs = np.random.choice(n, n, replace=True)

        # X
        if hasattr(X, "iloc"):
            Xb = X.iloc[idxs].reset_index(drop=True)
        else: 
            Xb = X[idxs]

        # y
        if hasattr(y, "iloc"):
            yb = y.iloc[idxs].reset_index(drop=True)
        else:
            yb = y[idxs]
        return Xb, yb

    def _most_common_label(self, y):
        return Counter(list(y)).most_common(1)[0][0]

    def fit(self, X, y):
        """
        Train multiple Decision Trees using bootstrapped samples and random feature subsets.
        """

        self._n_features_ = X.shape[1]
        ratio = float(self.feature_subsample_ratio)

        if ratio <= 0 or ratio > 1:
            feat_sub = int(np.sqrt(self._n_features_))
        else:
            feat_sub = max(1, int(round(ratio * self._n_features_)))
        self._feat_sub_count_ = feat_sub

        self.trees = []
        for _ in range(self.n_estimators):
            X_samp, y_samp = self._bootstrap_sample(X, y)

            tree = DecisionTree(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                feature_subsample=feat_sub,
            )
            tree.fit(X_samp, y_samp)
            self.trees.append(tree)
        return self

    def predict(self, X):
        """
        Predict class labels for all samples using majority voting from all trained trees.
        """

        preds = np.array([t.predict(X) for t in self.trees]).T
        return np.array([self._most_common_label(row) for row in preds])



In [ ]:
forest = RandomForest(n_estimators=60, min_samples_split=10, max_depth=6)
forest.fit(X_train, y_train)
y_pred_forest = forest.predict(X_test)

acc_forest = np.mean(y_pred_forest == y_test)
print(f"Custom Random Forest Accuracy: {acc_forest:.4f}")

compare_forest = pd.DataFrame({
    'Actual': y_test,
    'Tree_Pred': y_pred_tree,
    'Forest_Pred': y_pred_forest
})
compare_forest.head(7)


In [ ]:
print(f"Decision Tree Accuracy: {acc_tree:.4f}")
print(f"Random Forest Accuracy: {acc_forest:.4f}")


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_test, y_pred_tree))
print(classification_report(y_test, y_pred_tree))

print(confusion_matrix(y_test, y_pred_forest))
print(classification_report(y_test, y_pred_forest))


In [ ]:
# =============================
# TO BE IMPLEMENTED
# =============================

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
import numpy as np
import pandas as pd

# Ensure X_train / X_test are DataFrames (if not already)
if not isinstance(X_train, pd.DataFrame):
    X_train = pd.DataFrame(X_train)
    X_test = pd.DataFrame(X_test)

# Encode any categorical columns (if still string/object type)
for col in X_train.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])
    X_test[col] = le.transform(X_test[col])

# Scale numeric features, very important for SVMs
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# 1. Train and evaluate an SVM with a linear kernel
svm_linear = SVC(kernel='linear', random_state=42)
svm_linear.fit(X_train_scaled, y_train)
y_pred_linear = svm_linear.predict(X_test_scaled)
acc_linear = accuracy_score(y_test, y_pred_linear)
print(f"SVM (Linear Kernel) Accuracy: {acc_linear:.4f}")

# 2. Train and evaluate SVMs with an RBF kernel and different C values
for C_val in [0.1, 1, 10]:
    svm_rbf = SVC(kernel='rbf', C=C_val, gamma='scale', random_state=42)
    svm_rbf.fit(X_train_scaled, y_train)
    y_pred_rbf = svm_rbf.predict(X_test_scaled)
    acc_rbf = accuracy_score(y_test, y_pred_rbf)
    print(f"SVM (RBF Kernel, C={C_val}) Accuracy: {acc_rbf:.4f}")


In [ ]:
#Replace the accuracy variables with your defined variables for SVM accuracies

print(f"Custom Decision Tree Accuracy: {acc_tree:.4f}")
print(f"Custom Random Forest Accuracy: {acc_forest:.4f}")
print(f"Scikit-learn Linear SVM Accuracy: {acc_linear:.4f}")
print(f"Scikit-learn RBF SVM Accuracy {acc_rbf:.4f}")